# Build Chatbot
`11_chatbot.ipynb`
- https://python.langchain.com/docs/tutorials/chatbot/

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

True

In [32]:
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="Hi my name is jenny."),
    AIMessage(content="hello Jenny! ow can I help you."),
    HumanMessage(content="what is my name?"),
]

llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

res = llm.invoke(messages)

res.pretty_print()

================================== Ai Message ==================================

Your name is Jenny.


In [33]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, MessagesState, StateGraph

# Graph Builder
builder = StateGraph(state_schema=MessagesState)

# Node
def simple_node(state: MessagesState):
    res = llm.invoke(state['messages'])
    return {'messages': res}

builder.add_node('simple_node', simple_node)

# Edge (Node끼리 연결)
builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

# Memory (대화내역 기록)
memory = MemorySaver()

# Graph (그래프 생성)
graph = builder.compile(checkpointer=memory)

In [34]:
# 설정(conf, config, configuration -> 설정)

config = {'configurable': {'thread_id': 'abc123'}}  # 채팅방 아이디

graph.invoke({'messages': messages}, config=config)

{'messages': [HumanMessage(content='Hi my name is jenny.', additional_kwargs={}, response_metadata={}, id='bad8c39c-fbe0-4627-ae87-a958e711cb9e'),
  AIMessage(content='hello Jenny! ow can I help you.', additional_kwargs={}, response_metadata={}, id='13f20c74-3722-47ae-8622-91dcc0296531'),
  HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id='72838813-36eb-4ea0-af4c-ba30cb8056aa'),
  AIMessage(content='Your name is Jenny.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 36, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CDNXKoE52Y1Ai23YLxpDRMS6WmfUn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': N

In [35]:
config = {'configurable': {'thread_id': '가나다123'}} # 채팅방 아이디가 달라지면 대화 내용 기억 못함
message = [
    HumanMessage(content="say my name."),
]
graph.invoke({'messages': message}, config=config)

{'messages': [HumanMessage(content='say my name.', additional_kwargs={}, response_metadata={}, id='ebf2db6a-38a7-413b-bb59-e4cb37511e9a'),
  AIMessage(content="I'm sorry, but I don't know your name. Could you please tell me?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 11, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CDNXLMurvfnql9zORGoUoAnJsFPFM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ee288814-8b2c-4afd-8b26-eb8e0d788a94-0', usage_metadata={'input_tokens': 11, 'output_tokens': 16, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning':

In [36]:
import uuid

u_id = uuid.uuid1()
print(u_id)  # a0a765e1-8c4f-11f0-8098-d0c6378bea5b

config = {'configurable': {'thread_id': '가나다123'}} # 채팅방 아이디 -> 추후에는 UUID 형식으로 생성
messages = [
    HumanMessage(content="say my name."),
]
graph.invoke({'messages': messages}, config=config)

18ff752a-8c6a-11f0-a1f8-d0c6378bea5b


{'messages': [HumanMessage(content='say my name.', additional_kwargs={}, response_metadata={}, id='ebf2db6a-38a7-413b-bb59-e4cb37511e9a'),
  AIMessage(content="I'm sorry, but I don't know your name. Could you please tell me?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 11, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CDNXLMurvfnql9zORGoUoAnJsFPFM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ee288814-8b2c-4afd-8b26-eb8e0d788a94-0', usage_metadata={'input_tokens': 11, 'output_tokens': 16, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning':

## Langgraph + `PromptTemplate`

In [37]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '너는 유능한 어시스턴스야. 너의 능력을 최대한 활용해서 답을 해봐.'),
    MessagesPlaceholder(variable_name='messages') # 저장된 대화 내역
])

# 실행 예시
for msg in prompt_template.invoke({'messages': ['hi']}).messages:
    print(msg)

content='너는 유능한 어시스턴스야. 너의 능력을 최대한 활용해서 답을 해봐.' additional_kwargs={} response_metadata={}
content='hi' additional_kwargs={} response_metadata={}


## State 확장

In [ ]:
# 내장된 MessagesState를 확장해서 사용
class MyState(MessagesState):
    # 상속받아서 이미 key 'messages'는 있음
    # messages: Annotated[list[AnyMessage], add_messages]
    lang: str

builder = StateGraph(state_schema=MyState)

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '너는 유능한 어시스턴스야. 너의 능력을 최대한 활용해서 답을 해봐. {lang} 언어로 대답해줘.'),
    MessagesPlaceholder(variable_name='messages') # 저장된 대화 내역
])

def simple_node(state: MyState):
    # prompt 추가.
    chain = prompt_template | llm  # 체인 방식
    res = chain.invoke(state)

    return {'messages': res}

builder.add_node('simple_node', simple_node)

builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [39]:
config = {'configurable': {'thread_id': 'abc1'}} 
state = {
    'messages': [HumanMessage(content = 'say my name.')],
    'lang': 'Spanish'
}
res = graph.invoke(state, config)

for msg in res['messages']:
    msg.pretty_print()

================================ Human Message =================================

say my name.
================================== Ai Message ==================================

Lo siento, no tengo acceso a tu nombre. ¿Podrías decírmelo?


## 대화기록 관리하기
대화 내역을 관리하지 않으면, LLM의 컨텍스트 윈도우(입력 최대치)를 넘어가 버림.

In [41]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

# trim - 정리하다
trimmer = trim_messages(
    strategy='last',     # 최신 메시지들을
    max_tokens=65,        # 최대 65토큰까지만 허용
    token_counter=llm,   # llm 모델에 맞춰서 토큰 세고
    include_system=True, # system 프롬프트는 포함(정리X)
    allow_partial=False, # 메시지 중간에서 자르지는 말고
    start_on='human'     # 잘린 메세지의 첫번째는 사람 메시지가 되도록
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)  # 시스템메시지 -> 2+2 부터 등장

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [ ]:
class MyState(MessagesState):
    # messages: ~~
    lang: str


builder = StateGraph(state_schema=MyState)

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '너는 유능한 어시스턴트야. 너의 능력을 최대한 활용해서 답을 해봐. {lang} 언어로 답해.'),
    MessagesPlaceholder(variable_name='messages')  # 모든 저장된 대화 내용(최신것 포함)
])


def simple_node(state: MyState):
    # 메세지 정리 -> 프롬프트 생성 -> LLM 답변
    # print('정리 전 메시지 개수: ', len(state['messages']))
    trimmed_messages = trimmer.invoke(state['messages'])
    # print('정리 후 메시지 개수: ', len(trimmed_messages))
    
    # 체인 생성
    chain = prompt_template | llm
    
    # 정리된 메세지로 state 교체 후, 체인 실행
    state['messages'] = trimmed_messages
    res = chain.invoke(state)

    return {'messages': [res]}


builder.add_node('simple_node', simple_node)

builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [47]:
config = {'configurable': {'thread_id': 'abc1'}} 
state = {
    'messages': [HumanMessage(content='내 상태는 어때?')],
    'lang': '한국어'
}

# res = graph.invoke(state, config)

# # 채팅 저장은 잘 되지만, 대화내용이 잘려서 들어가는걸 확인 가능!
# for msg in res['messages']:
#     msg.pretty_print()

for chunk, metadate in graph.stream(state, config, stream_mode='messages'):
    print(chunk.content, end='|')

정리 전 메시지 개수:  9
정리 후 메시지 개수:  1
********************************************************
================================ Human Message =================================

내 상태는 어때?
	 None
********************************************************
|현재| 당신|의| 상태|에| 대해| 구|체|적인| 정보|가| 없|어서| 정확|히| 말씀|드|리|기| 어렵|습니다|.| 몸| 상태|,| 감|정| 상태|,| 또는| 다른| 어떤| 면|에| 대해| 말씀|하|시는| 건|가|요|?| 좀| 더| 구|체|적으로| 알려|주|시면| 도움|을| 드|릴| 수| 있을| 것| 같|아요|.||